# Entrega – Análisis Comparativo: V1 vs V2
**Agente Connect-4 | Fundamentos de Inteligencia Artificial**  
**Cristian Manuel Castañeda Gutiérrez | Universidad de La Sabana**

---

Este notebook compara dos versiones del agente:

| Versión | Archivo | Estrategia |
|---------|---------|------------|
| **V1** | `policy.py` | MCTS/UCT puro con heurística táctica (1 paso) |
| **V2** | `policy_qValues.py` | MCTS/UCT adversarial + Q-table persistente + rollouts guiados |

### Experimentos
1. W/D/L vs Agente Aleatorio (barras agrupadas y apiladas)
2. Barrido de simulaciones (línea + área)
3. Auto-partida / Self-Play
4. Enfrentamiento directo V1 vs V2 (gráfico de pastel)
5. Impacto del Q-Guided Rollout
6. Curva de aprendizaje de V2 (media móvil + acumulada)
7. Preferencia de columnas (distribución de movimientos)
8. Resumen comparativo final
9. V2 vs Grupo A – oponente competitivo (datos reales)

In [ ]:
import sys, os, warnings, contextlib, io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# Localizar la raíz del proyecto (funciona desde /groups/Cristian/ o desde la raíz)
ROOT = None
for _candidate in ['.', '../..', os.path.join(os.getcwd(), '..', '..')]:
    _abs = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_abs, 'connect4')):
        ROOT = _abs
        break
if ROOT is None:
    raise RuntimeError('No se encontró la raíz del proyecto. Ejecuta desde la carpeta connect-four-agents.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from groups.Cristian.policy import Cristian as V1
from groups.Cristian.policy_qValues import Cristian as V2

print(f'ROOT = {ROOT}')
print('Agentes cargados correctamente')

COLORS = {
    'win':  '#2ecc71',
    'draw': '#f39c12',
    'loss': '#e74c3c',
    'v1':   '#3498db',
    'v2':   '#9b59b6',
    'line': '#1abc9c',
    'bg':   '#f8f9fa',
}

In [ ]:
# ─── Funciones auxiliares ───────────────────────────────────────────────────

class RandomAgent(Policy):
    """Agente de referencia: columna válida al azar."""
    def mount(self, **_): pass
    def act(self, s: np.ndarray) -> int:
        cols = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.default_rng().choice(cols))


def play_game(red, yellow) -> int:
    """Una partida completa; retorna ganador (-1, 1) o 0."""
    rng = np.random.default_rng()
    state = ConnectState()
    while not state.is_final():
        col = red.act(state.board) if state.player == -1 else yellow.act(state.board)
        if not state.is_applicable(col):
            col = int(rng.choice(state.get_free_cols()))
        state = state.transition(col)
    return state.get_winner()


def matchup(agent_a, agent_b, n: int = 20):
    """
    n partidas como Rojo + n como Amarillo.
    Retorna (wr, dr, lr, wy, dy, ly) desde la perspectiva de agent_a.
    """
    wr = dr = lr = wy = dy = ly = 0
    for _ in range(n):
        agent_a.mount(); agent_b.mount()
        with contextlib.redirect_stdout(io.StringIO()):
            w = play_game(agent_a, agent_b)
        if w == -1: wr += 1
        elif w == 0: dr += 1
        else: lr += 1
    for _ in range(n):
        agent_a.mount(); agent_b.mount()
        with contextlib.redirect_stdout(io.StringIO()):
            w = play_game(agent_b, agent_a)
        if w == 1: wy += 1
        elif w == 0: dy += 1
        else: ly += 1
    return wr, dr, lr, wy, dy, ly


def pct(n, total): return n / total * 100 if total else 0

print('Funciones auxiliares listas.')

## Experimento 1 – W/D/L contra Agente Aleatorio
Evaluamos cuántas veces cada versión **gana, empata y pierde** jugando como Rojo y como Amarillo frente a un agente que elige columna al azar.

In [ ]:
N1 = 20
rand = RandomAgent()

print('Corriendo V1 vs Aleatorio...')
r1 = matchup(V1(simulations=500), rand, N1)
print('Corriendo V2 vs Aleatorio...')
r2 = matchup(V2(simulations=500), rand, N1)

print(f'\nV1  Rojo    : W={r1[0]:>2}  D={r1[1]:>2}  L={r1[2]:>2}  ({pct(r1[0],N1):.0f}% victorias)')
print(f'V1  Amarillo: W={r1[3]:>2}  D={r1[4]:>2}  L={r1[5]:>2}  ({pct(r1[3],N1):.0f}% victorias)')
print(f'V2  Rojo    : W={r2[0]:>2}  D={r2[1]:>2}  L={r2[2]:>2}  ({pct(r2[0],N1):.0f}% victorias)')
print(f'V2  Amarillo: W={r2[3]:>2}  D={r2[4]:>2}  L={r2[5]:>2}  ({pct(r2[3],N1):.0f}% victorias)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 1 – W/D/L vs Agente Aleatorio', fontsize=14, fontweight='bold')

agents   = ['V1\nRojo', 'V1\nAmarillo', 'V2\nRojo', 'V2\nAmarillo']
wins_    = [r1[0], r1[3], r2[0], r2[3]]
draws_   = [r1[1], r1[4], r2[1], r2[4]]
losses_  = [r1[2], r1[5], r2[2], r2[5]]
x1 = np.arange(4)
bw = 0.25

# Barras agrupadas
ax = axes[0]
b_w = ax.bar(x1 - bw, wins_,   bw, label='Victoria', color=COLORS['win'],  zorder=3)
b_d = ax.bar(x1,      draws_,  bw, label='Empate',   color=COLORS['draw'], zorder=3)
b_l = ax.bar(x1 + bw, losses_, bw, label='Derrota',  color=COLORS['loss'], zorder=3)
ax.set_xticks(x1); ax.set_xticklabels(agents)
ax.set_ylabel('Partidas'); ax.set_title('Barras Agrupadas', fontsize=12)
ax.legend(); ax.set_ylim(0, N1 + 4); ax.grid(axis='y', alpha=0.3, zorder=0)
for bars in [b_w, b_d, b_l]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, str(int(h)),
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

# Barras apiladas en %
ax2 = axes[1]
wp = [pct(w, N1) for w in wins_]
dp = [pct(d, N1) for d in draws_]
lp = [pct(l, N1) for l in losses_]
p1 = ax2.bar(agents, wp,  color=COLORS['win'],  label='Victoria', zorder=3)
p2 = ax2.bar(agents, dp,  bottom=wp,             color=COLORS['draw'], label='Empate', zorder=3)
p3 = ax2.bar(agents, lp,  bottom=[a+b for a,b in zip(wp,dp)], color=COLORS['loss'], label='Derrota', zorder=3)
ax2.set_ylabel('%'); ax2.set_title('Barras Apiladas (%)', fontsize=12)
ax2.set_ylim(0, 115); ax2.legend(loc='lower right'); ax2.grid(axis='y', alpha=0.3, zorder=0)
for bar, v in zip(p1, wp):
    if v > 5:
        ax2.text(bar.get_x()+bar.get_width()/2, v/2, f'{v:.0f}%',
                 ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('exp1_vs_random.png', bbox_inches='tight')
plt.show()
print('Guardado: exp1_vs_random.png')

## Experimento 2 – Barrido de Simulaciones
¿Cuántas simulaciones necesita el agente para rendir bien?  
Testeamos `[50, 100, 200, 500, 1000]` simulaciones por movimiento.

In [ ]:
SIMS_LIST = [50, 100, 200, 500, 1000]
rand2 = RandomAgent()
sweep_v1, sweep_v2 = [], []
N2 = 10

for sims in SIMS_LIST:
    r_v1 = matchup(V1(simulations=sims), rand2, N2)
    r_v2 = matchup(V2(simulations=sims), rand2, N2)
    wr1 = pct(r_v1[0]+r_v1[3], N2*2)
    wr2 = pct(r_v2[0]+r_v2[3], N2*2)
    sweep_v1.append(wr1); sweep_v2.append(wr2)
    print(f'sims={sims:>4}  V1={wr1:.0f}%  V2={wr2:.0f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(SIMS_LIST, sweep_v1, 'o-', color=COLORS['v1'], lw=2.5, ms=9, label='V1 (MCTS puro)', zorder=3)
ax.plot(SIMS_LIST, sweep_v2, 's-', color=COLORS['v2'], lw=2.5, ms=9, label='V2 (MCTS + Q-table)', zorder=3)
ax.fill_between(SIMS_LIST, sweep_v1, alpha=0.12, color=COLORS['v1'])
ax.fill_between(SIMS_LIST, sweep_v2, alpha=0.12, color=COLORS['v2'])
ax.set_xscale('log')
ax.set_xticks(SIMS_LIST); ax.set_xticklabels(SIMS_LIST)
ax.set_xlabel('Simulaciones por movimiento (escala log)', fontsize=11)
ax.set_ylabel('Tasa de Victoria (%)', fontsize=11)
ax.set_title('Experimento 2 – Rendimiento vs Número de Simulaciones\n(vs Agente Aleatorio, 10 partidas/color)', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3); ax.set_ylim(0, 108)
for x, y1, y2 in zip(SIMS_LIST, sweep_v1, sweep_v2):
    ax.annotate(f'{y1:.0f}%', (x, y1), textcoords='offset points', xytext=(0, 9),
                ha='center', color=COLORS['v1'], fontsize=9, fontweight='bold')
    ax.annotate(f'{y2:.0f}%', (x, y2), textcoords='offset points', xytext=(0, -16),
                ha='center', color=COLORS['v2'], fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('exp2_simulations.png', bbox_inches='tight')
plt.show()

## Experimento 3 – Auto-Partida (Self-Play)
En self-play simétrico esperamos ~50% para cada color.  
Una ventaja consistente de un color indica **sesgo de posición**.

In [ ]:
N3 = 15

print('V1 self-play...')
sp1 = matchup(V1(simulations=500), V1(simulations=500), N3)
print('V2 self-play...')
sp2 = matchup(V2(simulations=500), V2(simulations=500), N3)

for name, r in [('V1', sp1), ('V2', sp2)]:
    print(f'\n{name} self-play ({N3} partidas/rol):')
    print(f'  Rojo    : W={r[0]} D={r[1]} L={r[2]}')
    print(f'  Amarillo: W={r[3]} D={r[4]} L={r[5]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Experimento 3 – Self-Play (Simetría de Posición)', fontsize=14, fontweight='bold')

for ax, name, r in zip(axes, ['V1', 'V2'], [sp1, sp2]):
    cats  = ['Rojo gana', 'Empate', 'Amarillo gana']
    vals  = [r[0] + r[5], r[1] + r[4], r[2] + r[3]]
    clrs  = ['#e74c3c', '#f39c12', '#f1c40f']
    bars  = ax.bar(cats, vals, color=clrs, edgecolor='white', linewidth=1.5, zorder=3)
    ax.set_title(f'{name} Self-Play  ({N3} partidas/rol)', fontsize=12)
    ax.set_ylabel('Partidas totales')
    ax.set_ylim(0, N3 * 2 + 4)
    ax.axhline(N3 * 2 / 3, color='gray', linestyle='--', alpha=0.6, label='Balance esperado')
    ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + 0.3, str(v),
                ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('exp3_selfplay.png', bbox_inches='tight')
plt.show()

## Experimento 4 – Enfrentamiento Directo V1 vs V2
¿Cuánto mejora la Q-table? Enfrentamos ambas versiones directamente.

In [ ]:
N4 = 20
print('V2 vs V1 (directo)...')
r4 = matchup(V2(simulations=500), V1(simulations=500), N4)

total_v2 = r4[0] + r4[3]
total_v1 = r4[2] + r4[5]
draws4   = r4[1] + r4[4]
print(f'\nV2: {total_v2}/{N4*2} victorias ({pct(total_v2, N4*2):.1f}%)')
print(f'V1: {total_v1}/{N4*2} victorias ({pct(total_v1, N4*2):.1f}%)')
print(f'Empates: {draws4}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Experimento 4 – Enfrentamiento Directo V1 vs V2', fontsize=14, fontweight='bold')

for ax, role, data in zip(axes, ['V2 como Rojo', 'V2 como Amarillo'], [r4[:3], r4[3:]]):
    labels_pie = ['V2 gana', 'Empate', 'V1 gana']
    colors_pie = [COLORS['v2'], COLORS['draw'], COLORS['v1']]
    explode    = (0.06, 0, 0.06)
    wedges, texts, autotexts = ax.pie(
        data, labels=labels_pie, colors=colors_pie,
        autopct='%1.0f%%', explode=explode, startangle=120,
        textprops={'fontsize': 11}, pctdistance=0.75
    )
    for at in autotexts:
        at.set_fontweight('bold'); at.set_fontsize(12)
    ax.set_title(f'{role}  ({N4} partidas)', fontsize=12)

plt.tight_layout()
plt.savefig('exp4_v1_vs_v2.png', bbox_inches='tight')
plt.show()

## Experimento 5 – Impacto del Q-Guided Rollout
V2 puede usar la Q-table para guiar los rollouts (`q_guided=True`) o ignorarla (`q_guided=False`).  
Medimos el impacto de esta característica.

In [ ]:
N5 = 20
rand5 = RandomAgent()

print('V2 q_guided=True  vs Aleatorio...')
rg  = matchup(V2(simulations=500, q_guided=True),  rand5, N5)
print('V2 q_guided=False vs Aleatorio...')
rng = matchup(V2(simulations=500, q_guided=False), rand5, N5)

for label, r in [('Q-Guided=True', rg), ('Q-Guided=False', rng)]:
    total = r[0]+r[3]
    print(f'  {label}: {total}/{N5*2} victorias ({pct(total,N5*2):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Experimento 5 – Q-Guided Rollout: ON vs OFF', fontsize=14, fontweight='bold')

roles_  = ['Rojo', 'Amarillo']
for ax, i, role in zip(axes, [0, 3], roles_):
    data_g  = [rg[i], rg[i+1], rg[i+2]]
    data_ng = [rng[i], rng[i+1], rng[i+2]]
    x5 = np.arange(3)
    w5 = 0.35
    cats = ['Victoria', 'Empate', 'Derrota']
    clrs = [COLORS['win'], COLORS['draw'], COLORS['loss']]
    for k, (dg, dng, cat, clr) in enumerate(zip(data_g, data_ng, cats, clrs)):
        ax.bar(k - w5/2, dg,  w5, color=clr, alpha=0.9,  label=f'{cat} (ON)'  if k==0 else '_')
        ax.bar(k + w5/2, dng, w5, color=clr, alpha=0.45, label=f'{cat} (OFF)' if k==0 else '_',
               hatch='//')
    ax.set_xticks(x5); ax.set_xticklabels(cats)
    ax.set_title(f'Como {role}', fontsize=12)
    ax.set_ylabel('Partidas'); ax.set_ylim(0, N5 + 4)
    ax.grid(axis='y', alpha=0.3)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#555', label='Q-Guided ON'),
        Patch(facecolor='#aaa', hatch='//', label='Q-Guided OFF')
    ])

plt.tight_layout()
plt.savefig('exp5_qguided.png', bbox_inches='tight')
plt.show()

## Experimento 6 – Curva de Aprendizaje de V2
¿Mejora V2 a medida que acumula experiencia en la Q-table?  
Ejecutamos partidas **secuenciales** (sin reiniciar la Q-table entre partidas) y graficamos  
la tasa de victoria acumulada y la media móvil.

In [ ]:
N6 = 60
rand6   = RandomAgent()
v2_lrn  = V2(simulations=300, q_guided=True)
v2_lrn.mount()

wins_seq = []
for i in range(N6):
    rand6.mount()
    with contextlib.redirect_stdout(io.StringIO()):
        if i % 2 == 0:
            w = play_game(v2_lrn, rand6)
            won = 1 if w == -1 else 0
        else:
            w = play_game(rand6, v2_lrn)
            won = 1 if w == 1 else 0
    wins_seq.append(won)
    if (i+1) % 15 == 0:
        cum = sum(wins_seq)
        print(f'  Partida {i+1:>3}: victorias acum.={cum}/{i+1} ({pct(cum,i+1):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 6 – Curva de Aprendizaje de V2', fontsize=14, fontweight='bold')

# Media móvil
window = 10
roll   = np.convolve(wins_seq, np.ones(window)/window, mode='valid') * 100
x_roll = np.arange(window - 1, N6)

ax = axes[0]
ax.plot(x_roll, roll, color=COLORS['v2'], lw=2.5, label=f'Media móvil (ventana={window})')
ax.fill_between(x_roll, roll, alpha=0.18, color=COLORS['v2'])
ax.axhline(50, color='gray', ls='--', alpha=0.6, label='50% referencia')
ax.set_xlabel('Partidas jugadas'); ax.set_ylabel('Tasa de victoria (%)')
ax.set_title('Media Móvil', fontsize=12); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 108)

# Acumulada
cum_rate = [pct(sum(wins_seq[:i+1]), i+1) for i in range(N6)]
ax2 = axes[1]
ax2.plot(range(1, N6+1), cum_rate, color=COLORS['line'], lw=2.5, label='Tasa acumulada')
ax2.fill_between(range(1, N6+1), cum_rate, alpha=0.18, color=COLORS['line'])
ax2.axhline(50, color='gray', ls='--', alpha=0.6, label='50% referencia')
ax2.set_xlabel('Partidas jugadas'); ax2.set_ylabel('Tasa de victoria acumulada (%)')
ax2.set_title('Tasa Acumulada', fontsize=12); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 108)

plt.tight_layout()
plt.savefig('exp6_learning_curve.png', bbox_inches='tight')
plt.show()

## Experimento 7 – Preferencia de Columnas
¿Qué columnas elige cada agente con mayor frecuencia?  
La columna central (col 3) tiene la mayor conectividad y debería ser preferida.

In [ ]:
def collect_col_freqs(agent_cls, n_games=12, sims=300):
    counts = np.zeros(7, dtype=int)
    rand_  = RandomAgent()
    ag = agent_cls(simulations=sims)

    original_act = ag.act
    def tracked_act(s):
        c = original_act(s)
        counts[c] += 1
        return c
    ag.act = tracked_act

    for i in range(n_games):
        ag.mount(); rand_.mount()
        with contextlib.redirect_stdout(io.StringIO()):
            if i % 2 == 0:
                play_game(ag, rand_)
            else:
                play_game(rand_, ag)
    return counts

print('Recopilando movimientos V1...')
col_v1 = collect_col_freqs(V1)
print('Recopilando movimientos V2...')
col_v2 = collect_col_freqs(V2)
print('V1:', col_v1)
print('V2:', col_v2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Experimento 7 – Distribución de Movimientos por Columna', fontsize=14, fontweight='bold')

col_labels = [f'Col {i}' for i in range(7)]

for ax, counts, color, name in zip(axes,
                                    [col_v1, col_v2],
                                    [COLORS['v1'], COLORS['v2']],
                                    ['V1 (MCTS puro)', 'V2 (MCTS + Q-table)']):
    total = counts.sum()
    pcts  = counts / total * 100
    bar_colors = [color] * 7
    bar_colors[3] = '#e67e22'
    bars = ax.bar(col_labels, pcts, color=bar_colors, alpha=0.85,
                  edgecolor='white', linewidth=1.2, zorder=3)
    ax.axhline(100/7, color='gray', ls='--', alpha=0.6, label='Uniforme (14.3%)')
    ax.set_title(f'{name}', fontsize=12)
    ax.set_ylabel('%'); ax.set_ylim(0, max(pcts) * 1.35)
    ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)
    for bar, p in zip(bars, pcts):
        ax.text(bar.get_x()+bar.get_width()/2, p + 0.4, f'{p:.1f}%',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.text(bars[3].get_x()+bars[3].get_width()/2, -3, 'Centro',
            ha='center', va='top', fontsize=8, color='#e67e22', fontweight='bold')

plt.tight_layout()
plt.savefig('exp7_columns.png', bbox_inches='tight')
plt.show()

## Experimento 8 – Resumen Comparativo V1 vs V2

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

categories = [
    'vs Aleatorio\n(Rojo)',
    'vs Aleatorio\n(Amarillo)',
    'Directo\n(como Rojo)',
    'Directo\n(como Amarillo)',
]
v1_scores = [
    pct(r1[0], N1),
    pct(r1[3], N1),
    pct(r4[2], N4),
    pct(r4[5], N4),
]
v2_scores = [
    pct(r2[0], N1),
    pct(r2[3], N1),
    pct(r4[0], N4),
    pct(r4[3], N4),
]

x8 = np.arange(len(categories))
w8 = 0.35
bars1 = ax.bar(x8 - w8/2, v1_scores, w8, color=COLORS['v1'], label='V1 (MCTS puro)', alpha=0.88, zorder=3)
bars2 = ax.bar(x8 + w8/2, v2_scores, w8, color=COLORS['v2'], label='V2 (MCTS + Q-table)', alpha=0.88, zorder=3)
ax.axhline(50, color='gray', ls='--', alpha=0.5, label='50% referencia')
ax.set_xticks(x8); ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Tasa de Victoria (%)', fontsize=11); ax.set_ylim(0, 115)
ax.set_title('Resumen – Comparación General V1 vs V2', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3, zorder=0)

for bars, scores in [(bars1, v1_scores), (bars2, v2_scores)]:
    for bar, v in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, v + 1.5, f'{v:.0f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('exp8_resumen.png', bbox_inches='tight')
plt.show()

## Experimento 9 – V2 vs Grupo A (Oponente Competitivo Real)

El **Grupo A** implementa MCTS con tres mejoras avanzadas sobre V2:
- **Reutilización del árbol**: conserva el subárbol del turno anterior (grandchild reuse).
- **Sesgo progresivo (Progressive Bias)**: favorece columnas cercanas al centro en UCB.
- **Presupuesto de tiempo**: 60 s totales por partida → miles de simulaciones por turno.

> Resultados de una ejecución real con `validate_vs_groupA.py` (20 partidas/color).  
> No se re-ejecutan aquí para ahorrar tiempo de cómputo.

In [ ]:
# ── Datos reales del enfrentamiento (20 partidas/color) ────────────────────
N9 = 20

v2_red_w, v2_red_d, v2_red_l = 6,  0, 14   # V2 como Rojo   → 30%
v2_yel_w, v2_yel_d, v2_yel_l = 4,  0, 16   # V2 como Amarillo → 20%

ga_red_w  = N9 - v2_yel_w - v2_yel_d        # Grupo A como Rojo gana
ga_yel_w  = N9 - v2_red_w - v2_red_d        # Grupo A como Amarillo gana
total_v2  = v2_red_w + v2_yel_w             # 10 / 40
total_ga  = ga_red_w + ga_yel_w             # 30 / 40

print('Resultados reales – V2 vs Grupo A (20 partidas/color):')
print(f'  V2   Rojo    : W={v2_red_w:>2}  D={v2_red_d}  L={v2_red_l}  ({pct(v2_red_w,N9):.0f}%)')
print(f'  V2   Amarillo: W={v2_yel_w:>2}  D={v2_yel_d}  L={v2_yel_l}  ({pct(v2_yel_w,N9):.0f}%)')
print(f'  V2   TOTAL   : {total_v2}/{N9*2}  ({pct(total_v2,N9*2):.1f}%)')
print()
print(f'  Grupo A Rojo    : W={ga_red_w:>2}  ({pct(ga_red_w,N9):.0f}%)')
print(f'  Grupo A Amarillo: W={ga_yel_w:>2}  ({pct(ga_yel_w,N9):.0f}%)')
print(f'  Grupo A TOTAL   : {total_ga}/{N9*2}  ({pct(total_ga,N9*2):.1f}%)')

In [ ]:
fig = plt.figure(figsize=(15, 5))
fig.suptitle('Experimento 9 – V2 vs Grupo A  (20 partidas/color)', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# ── Panel izq: barras agrupadas W/D/L por rol ─────────────────────────────
ax0 = fig.add_subplot(gs[0])
roles_lbl = ['V2\n(Rojo)', 'V2\n(Amarillo)']
x0 = np.arange(2); bw0 = 0.25
w_vals = [v2_red_w, v2_yel_w]
d_vals = [v2_red_d, v2_yel_d]
l_vals = [v2_red_l, v2_yel_l]
ax0.bar(x0 - bw0, w_vals, bw0, color=COLORS['win'],  label='Victoria', zorder=3)
ax0.bar(x0,       d_vals, bw0, color=COLORS['draw'], label='Empate',   zorder=3)
ax0.bar(x0 + bw0, l_vals, bw0, color=COLORS['loss'], label='Derrota',  zorder=3)
ax0.set_xticks(x0); ax0.set_xticklabels(roles_lbl)
ax0.set_ylabel('Partidas'); ax0.set_title('W/D/L de V2 por Rol', fontsize=11)
ax0.legend(fontsize=9); ax0.set_ylim(0, N9 + 4)
ax0.grid(axis='y', alpha=0.3, zorder=0)
ax0.axhline(N9/2, color='gray', ls=':', alpha=0.5, label='50% ref.')
for x_, vals in [(x0 - bw0, w_vals), (x0, d_vals), (x0 + bw0, l_vals)]:
    for xi, v in zip(x_, vals):
        if v > 0:
            ax0.text(xi, v + 0.3, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Panel centro: pastel total ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
pie_vals   = [total_v2, total_ga]
pie_labels = [f'V2\n{total_v2} victorias\n({pct(total_v2,N9*2):.0f}%)',
              f'Grupo A\n{total_ga} victorias\n({pct(total_ga,N9*2):.0f}%)']
wedges, texts = ax1.pie(pie_vals, labels=pie_labels,
                         colors=[COLORS['v2'], '#e74c3c'],
                         startangle=90, explode=(0.06, 0),
                         textprops={'fontsize': 11})
ax1.set_title('Total de Victorias\n(40 partidas)', fontsize=11)

# ── Panel der: comparación de capacidades ────────────────────────────────
ax2 = fig.add_subplot(gs[2])
features   = ['Sims/turno\n(relativo)', 'Reutiliza\nárbol', 'Sesgo\nprogresivo', 'Q-table\npersistente']
score_v2   = [1.0, 0.0, 0.0, 1.0]
score_ga   = [4.0, 1.0, 1.0, 0.0]   # Grupo A hace ~4-6x más sims efectivas por turno
x2 = np.arange(len(features)); bw2 = 0.35
ax2.bar(x2 - bw2/2, score_v2, bw2, color=COLORS['v2'], label='V2',      alpha=0.88, zorder=3)
ax2.bar(x2 + bw2/2, score_ga, bw2, color='#e74c3c',    label='Grupo A', alpha=0.88, zorder=3)
ax2.set_xticks(x2); ax2.set_xticklabels(features, fontsize=8)
ax2.set_title('Capacidades Técnicas\n(escala relativa)', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3, zorder=0)
ax2.set_ylabel('Capacidad relativa')

plt.savefig('exp9_vs_groupA.png', bbox_inches='tight')
plt.show()
print('Guardado: exp9_vs_groupA.png')

### ¿Por qué pierde V2 contra el Grupo A?

| Factor | Grupo A | V2 | Impacto |
|--------|---------|----|---------|
| **Presupuesto de búsqueda** | 60 s totales → ~3000-5000 sims/turno | 500 sims fijas/turno | **6-10× más búsqueda** |
| **Reutilización del árbol** | Sí — conserva subárbol (nieto) | No — árbol nuevo cada turno | Sims del turno anterior no se pierden |
| **Sesgo progresivo UCB** | Sí — premia columna central | No | Exploración inicial más informada |
| **Q-table persistente** | No | Sí | Aprendizaje entre sesiones (único de V2) |

**Conclusión:** el cuello de botella es el **presupuesto de simulaciones efectivas**, no la calidad del rollout.  
Con reutilización de árbol, el Grupo A acumula exploración de turnos anteriores; V2 descarta ese trabajo en cada `mount()`.  
La Q-table de V2 es valiosa pero **no compensa la ventaja de búsqueda de 6-10×** del Grupo A.

## Análisis de Debilidades

### V1 – MCTS puro
| Debilidad | Evidencia | Causa |
|-----------|-----------|-------|
| Sin memoria entre partidas | Exp. 1: rendimiento estático | Cada `mount()` resetea el árbol |
| Rollouts completamente aleatorios | Exp. 2: curva de mejora limitada | Sin guía heurística en simulación |
| Sensible al número de simulaciones | Exp. 2: caída notable en sims bajas | Sin reutilización de cómputo |

### V2 – MCTS + Q-table
| Debilidad | Evidencia | Causa |
|-----------|-----------|-------|
| Pierde 75% vs Grupo A | Exp. 9: 10/40 victorias | Solo 500 sims fijas vs miles del rival |
| Q-table con ruido temprano | Exp. 6: alta varianza inicial | Pocas actualizaciones → estimaciones pobres |
| Q-values no generalizan | Sin feature engineering | Clave = bytes exactos del tablero |
| Overhead de I/O | Medición de turno | `pickle.dump` cada N movimientos |

## Conclusiones

### Principales hallazgos

1. **Ambos agentes dominan al aleatorio** (>90% de victorias con 500 simulaciones),  
   confirmando que MCTS es una estrategia sólida para Connect-4.

2. **V2 supera a V1** en el enfrentamiento directo gracias a:
   - Backpropagación adversarial (negamax-style): UCB1 correcto para ambos jugadores.
   - Q-table persistente: experiencia acumulada guía los rollouts.
   - Heurística táctica integrada: win-now/block-opponent sin coste extra.

3. **V2 pierde 75% contra el Grupo A** (Exp. 9): el Grupo A usa MCTS basado en tiempo  
   con reutilización de árbol, logrando 6-10× más simulaciones efectivas por turno.

4. **La columna central** (col 3) es la más elegida por ambos agentes,  
   coincidiendo con el conocimiento estratégico de Connect-4.

5. **El número de simulaciones** tiene rendimientos marginales decrecientes:  
   el mayor salto ocurre entre 50 y 200 sims; de 500 en adelante la mejora es pequeña.

---

## Mejoras Propuestas

### Mejora 1 – Reutilización del árbol MCTS entre turnos
**Motivación:** el árbol descartado al final de cada turno tiene nodos ya explorados  
que representan trabajo computacional perdido. El Exp. 9 muestra que esta es la  
diferencia crítica contra el Grupo A.  
**Implementación:** conservar el subárbol del movimiento elegido y el del rival (nieto),  
reutilizarlo como raíz en el turno siguiente.  
**Ganancia esperada:** multiplicar las simulaciones efectivas por turno 3-5×.

### Mejora 2 – Cambiar a presupuesto de tiempo (como Grupo A)
**Motivación:** con 500 sims fijas, V2 es lento en posiciones simples y limitado en  
posiciones complejas. Un presupuesto de tiempo usa el cómputo disponible de forma  
adaptativa.  
**Implementación:** reemplazar `for _ in range(self.simulations)` por  
`while (time.time() - start) < time_budget`.  
**Ganancia esperada:** igualar el nivel de búsqueda del Grupo A.

### Mejora 3 – Tablas de transposición (Zobrist hashing)
**Motivación:** la Q-table usa `board.tobytes()` como clave — estados similares no  
comparten información. Un hash de Zobrist identificaría tableros equivalentes.  
**Implementación:** hash incremental XOR al aplicar cada movimiento; mapear al  
mismo Q-value estados que lleguen al mismo tablero por caminos distintos.  
**Ganancia esperada:** Q-table más densa y con estimaciones convergentes más rápido.